# Challenge 4 · Neural Operators

[Start Here](../../Start_Here.ipynb) · Previous: [Climate](../03_climate/Challenge_3_Climate_Modeling.ipynb)

The earlier PINNs take coordinates, sometimes with a few problem parameters. Here, the input is an entire forcing field and the output is its solution field. Learn this map across many forcing fields, then compare FNO, AFNO and PINO on the same data.

| Level | File to edit | Configuration | What you will learn |
|---|---|---|---|
| 1 | [fno_physicsnemo_l1.py](fno_physicsnemo_l1.py) | [config_FNO.yaml](conf/config_FNO.yaml) | Build independent data splits and an FNO model |
| 2 | [fno_physicsnemo_l2.py](fno_physicsnemo_l2.py) | [config_AFNO.yaml](conf/config_AFNO.yaml) | Relate AFNO patches to the field grid |
| 3 | [fno_physicsnemo_l3.py](fno_physicsnemo_l3.py) | [config_PINO.yaml](conf/config_PINO.yaml) | Add a symbolic PDE residual to training |

The shared training loop is in [operator_training.py](operator_training.py). Follow `optimizer.zero_grad → forward → loss.backward → optimizer.step` to see how either the data loss or the combined loss updates the model.

Build each model using separate training, validation, and test splits. Compare solution error with equation error. The results are practice feedback; official competition scores and ranking rules have not been defined. See the [assessment guide](../../ETC/course_materials/ASSESSMENT.md) for metric definitions.


## Problem: from forcing to solution

$$u(x,y)-\Delta u(x,y)=f(x,y),\qquad (x,y)\in[0,1]^2$$

with periodic boundary conditions. The learned operator is $\mathcal G:f\mapsto u=(1-\Delta)^{-1}f$.
The discrete grid contains $x_i=i/N$, $y_j=j/N$, $i,j=0,\ldots,N-1$, excluding the duplicate periodic endpoint at 1.
This gives spacing $1/N$ and matches spectral differentiation.

The input and output have different scales: high-frequency forcing coefficients are divided by
$1+(2\pi)^2(k^2+l^2)$. Compute separate input/output means and standard deviations from training data only, use the same values for validation and test data, and restore physical units before computing the PDE residual.

All three levels use the same synthetic reaction-diffusion fields so you can compare their predictions and equation residuals on a common problem.

## Run this Challenge

Use PhysicsNeMo 2.2.2 in the notebook kernel's Python environment.

1. Run setup in student mode (`USE_REFERENCE = False`). Unfinished factories should stop with a message. Set `True` explicitly only for an instructor demonstration; it bypasses your edits.
2. Complete the `FIXME` functions in the linked Python file. Save with Ctrl+S / Command+S; editing an example in this notebook does not change the program. Unfinished functions report what is missing.
3. Run the data cell once, then the level's training and inspection cells. Each attempt writes to a new output directory. After changing code or mode, rerun the relevant training and result cells; no server restart is needed.
4. Choose `STEPS` and `DEVICE` in setup. A two-step CPU run checks execution; assess learning with the held-out errors below. `DEVICE = "cuda"` requires a GPU. Notebook steps override the configuration's `training.steps = 10000`.

The default configurations require a 64×64 dataset. Check the data and output paths in setup, and the command and mode printed by each training cell. The result table compares errors before and after training. Open "Full metrics and run settings" for the complete record.

Local plots are practice feedback, not submitted scores. Register your nickname in **Submit your code** below, then use that panel to send your saved implementation and view the judge's results in this notebook. Instructor demonstrations cannot be submitted.


In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
from uuid import uuid4

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "02_challenges/04_neural_operators/fno_physicsnemo_l1.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the bootcamp repository.")
LESSON_DIR = ROOT / "02_challenges/04_neural_operators"
sys.path.insert(0, str(ROOT))
from ETC.runtime.notebook import show_results, validate_settings
from ETC.runtime.notebook import require_current_mode as check_saved_mode

DEVICE = os.environ.get("AI4SCI_DEVICE", "auto")  # also accepts "cpu" or "cuda"
STEPS = int(os.environ.get("AI4SCI_STEPS", "200"))
SEED = 42
USE_REFERENCE = False  # Student mode. Set True only for an instructor demonstration.
# Server AI4SCI_REFERENCE settings do not change this student default.
JUDGE_URL = os.environ.get("AI4SCI_JUDGE_URL", "")  # Optional API URL; private launch configuration is also supported.
DATA_DIR = Path(os.environ.get("AI4SCI_DATA_DIR", str(LESSON_DIR / "datasets" / "Reaction_Diffusion"))).expanduser().resolve()
OUTPUT_BASE = Path(os.environ.get("AI4SCI_OUTPUT_DIR", str(LESSON_DIR / "outputs"))).expanduser().resolve()
RUN_DIRS = {}  # Latest attempt per level; older artifacts remain on disk.
RUN_COMPLETED = {}

def show_mode():
    validate_settings(DEVICE, STEPS, USE_REFERENCE)
    print("Mode: INSTRUCTOR REFERENCE; student functions are bypassed." if USE_REFERENCE
          else "Mode: STUDENT; saved exercise functions will run.")

def run_script(script, *arguments):
    command = [sys.executable, str(LESSON_DIR / script), *map(str, arguments)]
    subprocess.run(command, cwd=LESSON_DIR, check=True)

def require_current_mode(metrics, level):
    check_saved_mode(metrics, USE_REFERENCE, level)

def inspect_level(level):
    if not RUN_COMPLETED.get(level, False):
        raise RuntimeError(f"The current Level {level} training run has not completed.")
    return show_results(RUN_DIRS[level], reference=USE_REFERENCE)

show_mode()
print({"python": sys.executable, "lesson": str(LESSON_DIR), "device": DEVICE,
       "steps": STEPS, "output_base": str(OUTPUT_BASE)})

from ETC.runtime.submission import show_submission_panel, show_submission_controls
show_submission_panel("4", reference=USE_REFERENCE, judge_url=JUDGE_URL)


## Step 0: Data generation

[generate_data.py](generate_data.py) constructs forcing fields from random sine/cosine tensor-product Fourier modes up to $K=6$.
For each coefficient $a_{kl}$ multiplying a basis function $\phi_{kl}$:

$$f=\sum_{k,l} a_{kl}\phi_{kl},\qquad
u=\sum_{k,l}\frac{a_{kl}}{1+(2\pi)^2(k^2+l^2)}\phi_{kl}.$$

The tensor product combines modes along both spatial axes. All modes lie below the Nyquist frequency; an independent FFT residual checks the analytical pairs before float32 storage.

Default files, with shape `[samples, 1, 64, 64]`:

| Split | Samples | Role |
|---|---:|---|
| `train.hdf5` | 8,000 | Gradient updates and normalization statistics |
| `val.hdf5` | 1,000 | Development evaluation |
| `test.hdf5` | 1,000 | Held-out comparison; never used by the optimizer |

The three random streams are spawned independently from `--seed`. The generator writes a schema and split identity to HDF5 plus `manifest.json`.
The archived [ETC/legacy/Poisson_Fourier](../../ETC/legacy/Poisson_Fourier) files solve a different problem. Use the generated pairs for this exercise; the loader rejects the archived schema.

```bash
python generate_data.py --seed 42
```

To generate another dataset, choose a fresh `--output-dir` and use the corresponding training `--data-dir`.
For a quick CPU run, use `--train-samples 16 --val-samples 4 --test-samples 4` with the same 64×64 grid.
Reducing the grid also requires matching `data.grid_size`, FNO modes and AFNO patch sizes in a separate configuration.

In [ ]:
# Generate once, then reuse the same independent splits for all three levels.
required_files = [DATA_DIR / f"{split}.hdf5" for split in ("train", "val", "test")]
if all(path.exists() for path in required_files):
    sys.path.insert(0, str(LESSON_DIR))
    from operator_training import load_data
    load_data(DATA_DIR)
    print("Reusing validated reaction-diffusion dataset:", DATA_DIR)
else:
    run_script("generate_data.py", "--seed", SEED, "--output-dir", DATA_DIR)

## Level 1 · Fourier Neural Operator

FNO captures global correlations by Fourier transformation, learned multiplication of retained modes, inverse transformation, and a local skip/activation path:

$$v_{j+1}=\sigma\left(W_jv_j+\mathcal F^{-1}\big(R_j\cdot\mathcal F(v_j)\big)\right).$$

The FFT contributes $O(N\log N)$ work; total model cost also depends on channels, layers and retained modes.
`FNO` takes and returns tensors with shape `[batch, channels, height, width]`.

```python
from torch.utils.data import TensorDataset
from physicsnemo.models.fno import FNO

train_dataset = TensorDataset(normalized_f_train, normalized_u_train)
model = FNO(in_channels=1, out_channels=1, dimension=2,
            latent_channels=32, num_fno_layers=4, num_fno_modes=12,
            padding=0, coord_features=False)
prediction = model(normalized_f_batch)  # [batch, 1, N, N]
loss = (prediction - normalized_u_batch).square().mean()
optimizer.zero_grad(set_to_none=True)
loss.backward()
optimizer.step()
```

Implement `build_datasets` and `build_model` in [fno_physicsnemo_l1.py](fno_physicsnemo_l1.py), then save.
Use three separate `TensorDataset` objects. Preserve the supplied values and sample order: normalization has already been done. The output shape must equal the input shape.
The trainer checks all three datasets against the supplied splits and builds its own validation/test loaders. Swapped splits or changed test labels raise an error. These local checks catch implementation mistakes; they are not a secure competition grader.
Periodic data use zero padding width and no nonperiodic coordinate channels in this baseline.
The trainer performs normalization outside the model and restores physical units for evaluation.

In [ ]:
# Terminal equivalent from 02_challenges/04_neural_operators:
# python fno_physicsnemo_l1.py --device cpu --steps 200 --output-dir outputs/my-l1
# Add --reference for the instructor implementation.
RUN_COMPLETED[1] = False
result_dir = OUTPUT_BASE / f"level1-{uuid4().hex}"
RUN_DIRS[1] = result_dir
command = [sys.executable, str(LESSON_DIR / "fno_physicsnemo_l1.py"),
           "--device", DEVICE, "--steps", str(STEPS), "--seed", str(SEED),
           "--data-dir", str(DATA_DIR), "--output-dir", str(result_dir)]
if USE_REFERENCE:
    command.append("--reference")
show_mode()
print(command)
subprocess.run(command, cwd=LESSON_DIR, check=True)
RUN_COMPLETED[1] = True

In [ ]:
metrics_l1 = inspect_level(1)

## Level 2 · Adaptive Fourier Neural Operator

AFNO uses patch embedding, FFT over the patch grid, block-diagonal channel mixing with nonlinearities and sparsity, and reconstruction to the output grid.
Both FNO and AFNO learn their spectral transformations. Here the comparison is between FNO layers and AFNO's patch-based token mixing.

```python
from physicsnemo.models.afno import AFNO
model = AFNO(inp_shape=[64, 64], in_channels=1, out_channels=1,
             patch_size=[4, 4], embed_dim=64, depth=4, num_blocks=8)
prediction = model(normalized_f_batch)
```

Build the same separate datasets and instantiate `AFNO` in [fno_physicsnemo_l2.py](fno_physicsnemo_l2.py).
Check that both image dimensions are divisible by the patch dimensions. Do not crop the periodic square: cropping would change the domain and the PDE boundary conditions.
`embed_dim` must also be divisible by `num_blocks`.

Use the same training/validation/test split as Level 1. Compare held-out relative L2 error and runtime. How does changing patch size alter the token grid and the detail the model must reconstruct?

In [ ]:
# Terminal equivalent from 02_challenges/04_neural_operators:
# python fno_physicsnemo_l2.py --device cpu --steps 200 --output-dir outputs/my-l2
# Add --reference for the instructor implementation.
RUN_COMPLETED[2] = False
result_dir = OUTPUT_BASE / f"level2-{uuid4().hex}"
RUN_DIRS[2] = result_dir
command = [sys.executable, str(LESSON_DIR / "fno_physicsnemo_l2.py"),
           "--device", DEVICE, "--steps", str(STEPS), "--seed", str(SEED),
           "--data-dir", str(DATA_DIR), "--output-dir", str(result_dir)]
if USE_REFERENCE:
    command.append("--reference")
show_mode()
print(command)
subprocess.run(command, cwd=LESSON_DIR, check=True)
RUN_COMPLETED[2] = True

In [ ]:
metrics_l2 = inspect_level(2)

## Level 3 · Physics-Informed Neural Operator

PINO combines a neural-operator backbone with data and physics losses. Here the backbone is the same FNO as Level 1:

$$\mathcal L=\mathcal L_{\rm data}+\lambda\mathcal L_{\rm physics},\qquad
r=u_{\rm pred}-\Delta u_{\rm pred}-f.$$

The data loss compares normalized solutions. The physics loss is $\operatorname{mean}[(r/\sigma_f)^2]$, where $\sigma_f$ is the training forcing standard deviation. This scaling leaves the PDE unchanged. Residual metrics are reported in physical units.

Define the equation symbolically and evaluate its residual on the predicted physical field:

```python
from sympy import Function, Symbol
from physicsnemo.sym.eq.pde import PDE
from physicsnemo.sym.eq.phy_informer import PhysicsInformer
from operator_training import BatchedPhysicsInformer

class ReactionDiffusion(PDE):
    def __init__(self):
        self.dim = 2
        x, y = Symbol("x"), Symbol("y")
        u, f = Function("u")(x, y), Function("f")(x, y)
        self.equations = {"reaction_diffusion": u - u.diff(x, 2) - u.diff(y, 2) - f}

physics = BatchedPhysicsInformer(PhysicsInformer(
    required_outputs=["reaction_diffusion"], equations=ReactionDiffusion(),
    grad_method="spectral", bounds=[1.0, 1.0], device="cpu"))
residual = physics.forward({"u": predicted_u_physical, "f": forcing_physical})["reaction_diffusion"]
physics_loss = (residual / training_f_std).square().mean()
```

Complete the dataset, FNO and `ReactionDiffusionPDE.equations` functions in [fno_physicsnemo_l3.py](fno_physicsnemo_l3.py).
The provided `build_physics` connects this equation to spectral differentiation. The explicit training loop backpropagates through both losses into FNO.

For the periodic unit square, spectral second derivatives multiply Fourier coefficients by $-(2\pi k)^2$ and $-(2\pi l)^2$.
`BatchedPhysicsInformer` evaluates one scalar field at a time, keeping samples independent while retaining gradients through the residual.
The evaluator separately computes the full Laplacian with `torch.fft.fft2` and checks it agrees with `PhysicsInformer`.
When you change the physics-loss weight, measure both held-out solution error and PDE residual. Do they improve together, or is there a trade-off?

In [ ]:
# Terminal equivalent from 02_challenges/04_neural_operators:
# python fno_physicsnemo_l3.py --device cpu --steps 200 --output-dir outputs/my-l3
# Add --reference for the instructor implementation.
RUN_COMPLETED[3] = False
result_dir = OUTPUT_BASE / f"level3-{uuid4().hex}"
RUN_DIRS[3] = result_dir
command = [sys.executable, str(LESSON_DIR / "fno_physicsnemo_l3.py"),
           "--device", DEVICE, "--steps", str(STEPS), "--seed", str(SEED),
           "--data-dir", str(DATA_DIR), "--output-dir", str(result_dir)]
if USE_REFERENCE:
    command.append("--reference")
show_mode()
print(command)
subprocess.run(command, cwd=LESSON_DIR, check=True)
RUN_COMPLETED[3] = True

In [ ]:
metrics_l3 = inspect_level(3)

## Summary and comparison

| Aspect | FNO | AFNO | PINO in this lesson |
|---|---|---|---|
| Model | FNO spectral layers | AFNO patch/token mixing | Same FNO as Level 1 |
| Training objective | Data loss | Data loss | Data loss + PDE residual |
| PDE residual in training | No | No | `PhysicsInformer` |
| Validation | Held-out solution and residual metrics | Same | Same + independent residual agreement |

Every successful run writes `metrics.json`, `loss.csv`, `model.pt`, `predictions.npz`, and `preview.png` when matplotlib is available. The checkpoint includes the model settings and training-only normalization statistics.

Compare `test_relative_l2_before` and `test_relative_l2_after`, the physical-unit PDE RMSE, and the spatial error plot across levels. Lower values mean smaller errors, but solution and PDE errors may not improve together. Use the same held-out fields and training budget. Raw training losses are not directly comparable because the objectives differ.

Use validation results while choosing settings. The public test split is a final local comparison, not a hidden competition set; repeated tuning against it weakens the held-out interpretation. Official individual ranking will require a separately controlled evaluator and published rules.

The result viewer checks the `reference` flag saved in `metrics.json`. After changing `USE_REFERENCE`, rerun the levels you want to compare. Runs from another mode, or with an unknown mode, are excluded.

In [ ]:
for level in (1, 2, 3):
    if not RUN_COMPLETED.get(level, False):
        print(f"Level {level}: latest attempt incomplete; no previous result shown.")
        continue
    path = RUN_DIRS[level] / "metrics.json"
    if path.exists():
        result = json.loads(path.read_text())
        try:
            require_current_mode(result, level)
        except RuntimeError as error:
            print(error)
            continue
        print(result["method"], {
            "steps": result["steps"],
            "relative_l2_before": result["test_relative_l2_before"],
            "relative_l2_after": result["test_relative_l2_after"],
            "pde_rmse_fft": result["test"]["pde_rmse_fft"],
        })

## References and further reading

- [FNO: Li et al., Fourier Neural Operator for Parametric PDEs](https://arxiv.org/abs/2010.08895)
- [AFNO: Guibas et al., Adaptive Fourier Neural Operators](https://arxiv.org/abs/2111.13587)
- [PINO: Li et al., Physics-Informed Neural Operator](https://arxiv.org/abs/2111.03794)
- [Unified PhysicsNeMo repository](https://github.com/NVIDIA/physicsnemo)
- [Official FNO and AFNO API](https://docs.nvidia.com/physicsnemo/latest/physicsnemo/api/models/fnos.html)
- [Official PDE and PhysicsInformer API](https://docs.nvidia.com/physicsnemo/latest/physicsnemo/api/physicsnemo.sym.html)
- [Official PhysicsNeMo example catalog](https://docs.nvidia.com/physicsnemo/latest/physicsnemo/examples/index.html)

### Check your understanding

- What changes between learning one solution and learning a forcing-to-solution map?
- Why must normalization statistics come only from the training split?
- Predict the effect of changing AFNO patch size or PINO's physics-loss weight. Save and run one change, then compare solution error, runtime, and PDE residuals on the same data.

[Start Here](../../Start_Here.ipynb) · Previous: [Climate](../03_climate/Challenge_3_Climate_Modeling.ipynb)

## Submit your code

Save `build_datasets` and `build_model` in each completed Level. For Level 3, also complete `ReactionDiffusionPDE.__init__`. Run the cell below. Before your first submission, enter your **Nickname** and click **Register nickname**. This name is shared by Challenges 1-4 and appears on the public scoreboard. Then select the completed Levels and click **Submit code**. Queue status and scores appear here. Running the cell or Run All does not submit code.

The pilot accepts the original TensorDataset, FNO and AFNO constructor patterns, local assignments and explicit AFNO grid checks. It parses these functions without executing arbitrary Python. Model channels, grid and all supplied configuration settings must match the course model. For PINO, write the PDE using the supplied Symbol/Function declarations.

The server owns normalization, training and evaluation. Its pilot dataset has 64/16/16 train/validation/test samples on the same 64×64 grid, with a fixed seed different from local practice. It scores implementation checks, test relative L2, RMSE and independent FFT equation error. PINO also verifies PhysicsInformer against the FFT residual. The smaller dataset is for the scoring pilot, not the full lesson benchmark.

Include all Levels you want counted in this attempt; missing Levels count as zero. The best complete submission per Challenge is retained. All four Challenges now contribute to the provisional 400-point total. Official event rules still need calibration and approval.


In [ ]:
# Run All only opens these controls. Submission requires a button click.
show_submission_controls("4", LESSON_DIR,
    levels=(1,), reference=USE_REFERENCE, judge_url=JUDGE_URL)


--- 

Further resources: [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources). Community support: [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack).

---

# Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.
